In [ ]:
%cd /content
!rm -rf /content/LearnKG /content/LearnKG-ACL /content/LearnKG.zip
!wget -q -O LearnKG.zip https://github.com/thegreatvoid/LearnKG/archive/refs/heads/ACL.zip
!unzip -q LearnKG.zip && mv LearnKG-ACL LearnKG
%cd /content/LearnKG
!grep -c "rarest_word_zipf" kg_extract/concepts.py   # must print a number above 0; if it prints 0, Step 1 wasn't pushed

/content
/content/LearnKG
2


In [ ]:
!pip install -q pyvis spacy wordfreq
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 35.4 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/loca

In [ ]:
!pkill -9 ollama || true
!nohup ollama serve > /tmp/ollama.log 2>&1 &
!sleep 5 && curl -s http://127.0.0.1:11434/
!sleep 3 && grep -i "inference compute" /tmp/ollama.log   # should mention CUDA (the GPU)


Ollama is runningtime=2026-10-25T12:45:27.255Z level=INFO source=types.go:32 msg="inference compute" id=0 filter_id=0 library=CUDA compute=7.5 name=CUDA0 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:04.0 type=discrete total="14.6 GiB" available="14.5 GiB"


In [ ]:
!ollama pull qwen2.5:7b-instruct
!ollama pull llama3.1:8b
!time ollama run qwen2.5:7b-instruct "Say hello in one sentence."   # first load takes 1-3 min, don't interrupt




Hello there! How can I assist you today?


real	1m48.071s
user	0m0.072s
sys	0m0.095s


In [ ]:
!python -m kg_extract.run --data-dir ck12_geometry --model qwen2.5:7b-instruct --hubs 10 --samples 2 --out data_output/ck12_geometry_kgx_v2 2>&1 | tee /tmp/kgx.log


In [ ]:
#Run improved (chatgpt's suggestion)
!python -m kg_extract.run \
  --data-dir ck12_geometry \
  --model qwen2.5:7b-instruct \
  --hubs 10 \
  --per-concept 15 \
  --max-pairs 3000 \
  --samples 3 \
  --threshold 0.5 \
  --out data_output/ck12_geometry_experiment

[concepts] {'scored': 357, 'kept': 250} counters={'candidates_seen': 1214, 'chunks_llm_ok': 60, 'candidates_dropped_ungrounded': 16, 'candidates_dropped_weak_single_word': 40, 'candidates_dropped_weak_common_phrase': 20, 'candidates_dropped_min_freq': 8, 'concepts_kept': 203} (all candidates: data_output/ck12_geometry_experiment/concept_candidates.csv)
[pairs] 2685 candidate pairs


In [ ]:
#chatgpt exp
!python -m kg_eval.concept_quality \
  --kg-dir data_output/ck12_geometry_experiment \
  --judge-model llama3.1:8b \
  --subject geometry

In [ ]:
#chatgpt exp
!python -m kg_eval.prereq_gold \
  --kg-dir data_output/ck12_geometry_experiment

In [ ]:
%cd /content/LearnKG
!zip -r /content/ck12_geometry_experiment.zip data_output/ck12_geometry_experiment

In [ ]:
from google.colab import files
files.download('/content/ck12_geometry_experiment.zip')

In [ ]:
!python -m kg_eval.concept_quality --kg-dir data_output/ck12_geometry_kgx_v2 --judge-model llama3.1:8b --subject geometry
!python -m kg_eval.prereq_gold --kg-dir data_output/ck12_geometry_kgx_v2


In [ ]:
#Optional: to also build the hybrid version (your best AL-CPL result so far), run this before Cell 8 and render the _hybrid folder too:
!python -m kg_extract.hybrid --run data_output/ck12_geometry_kgx_v2 --out data_output/ck12_geometry_kgx_v2_hybrid

In [ ]:
from educational_pipeline.graph_builder import build_educational_graph
d = "data_output/ck12_geometry_kgx_v2"
build_educational_graph(f"{d}/concepts.csv", f"{d}/relations.csv", f"{d}/graph.html")
!zip -qr /content/kgx_v2.zip data_output/ck12_geometry_kgx_v2
from google.colab import files; files.download("/content/kgx_v2.zip")


NEW TEST BY CHATGPT TOP K 6

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving ck12_geometry_kgx_v2_hybrid.zip to ck12_geometry_kgx_v2_hybrid.zip


In [ ]:
%cd /content

!wget -q https://github.com/thegreatvoid/LearnKG/archive/refs/heads/ACL.zip -O /content/LearnKG.zip
!unzip -qo /content/LearnKG.zip -d /content
!mv /content/LearnKG-ACL /content/LearnKG

%cd /content/LearnKG

/content
/content/LearnKG


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/ck12_geometry_kgx_v2_hybrid.zip")
assert zip_path.exists(), f"ZIP not found: {zip_path}"

with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content/LearnKG")

!find /content/LearnKG/data_output -maxdepth 2 -type f | head -40

/content/LearnKG/data_output/ck12_geometry/relations.csv
/content/LearnKG/data_output/ck12_geometry/candidate_pairs.csv
/content/LearnKG/data_output/ck12_geometry/evaluation_report.json
/content/LearnKG/data_output/ck12_geometry/concepts.csv
/content/LearnKG/data_output/ck12_geometry/gold_prerequisite_pairs.csv
/content/LearnKG/data_output/stage_attrition_pairs_rule_v3.csv
/content/LearnKG/data_output/stage_attrition_pairs_kgx_v2.csv
/content/LearnKG/data_output/bigs_compare.json
/content/LearnKG/data_output/ck12_geometry_kgx_run1/relations.csv
/content/LearnKG/data_output/ck12_geometry_kgx_run1/run_meta.json
/content/LearnKG/data_output/ck12_geometry_kgx_run1/pair_scores.csv
/content/LearnKG/data_output/ck12_geometry_kgx_run1/concepts.csv
/content/LearnKG/data_output/eval_framework/REPORT.md
/content/LearnKG/data_output/reconciliation_report.json
/content/LearnKG/data_output/ck12_geometry_colab/relations.csv
/content/LearnKG/data_output/ck12_geometry_colab/concepts.csv
/content/LearnK

In [ ]:
%cd /content/LearnKG

!ls -la data_output/ck12_geometry_kgx_v2_hybrid
!ls -la data_input/ck12_geometry_cleaned

/content/LearnKG
total 3028
drwxr-xr-x  3 root root    4096 Oct 25 15:33 .
drwxr-xr-x 19 root root    4096 Oct 25 15:33 ..
-rw-r--r--  1 root root   61846 Oct 25 15:33 concepts.csv
drwxr-xr-x  2 root root    4096 Oct 25 15:33 kg_eval
-rw-r--r--  1 root root 2898869 Oct 25 15:33 pair_scores.csv
-rw-r--r--  1 root root  117849 Oct 25 15:33 relations.csv
-rw-r--r--  1 root root     261 Oct 25 15:33 run_meta.json
total 68
drwxr-xr-x 2 root root  4096 Oct 25 15:33 .
drwxr-xr-x 5 root root  4096 Oct 25 15:33 ..
-rw-r--r-- 1 root root 44745 Oct 25 15:33 geometry.pairs
-rw-r--r-- 1 root root 14063 Oct 25 15:33 geometry.preqs


In [ ]:
!python -m kg_eval.prereq_gold \
  --kg-dir data_output/ck12_geometry_kgx_v2_hybrid

Gold concepts matched to KG: 65/84;  pairs matched: 989/1595;  located in text: 1431

MAIN (matched & located pairs)  (n=989, positives=293, prevalence=0.296)
  predictor             P      R     F1  F1 95% CI          MCC  ROC-AUC  AUC 95% CI       PR-AUC
  kg_prereq         0.835  0.294  0.434  [0.37, 0.49]     0.402    0.635  [0.61, 0.66]      0.473
  kg_closure        0.815  0.874  0.843  [0.81, 0.88]     0.775    0.900  [0.88, 0.92]      0.768
  kg_any_relation   0.760  0.314  0.444  [0.38, 0.50]     0.379    0.637  [0.61, 0.66]      0.460
  kg_undirected     0.397  0.297  0.340  [0.29, 0.39]     0.118    0.554  [0.53, 0.58]      0.326
  kg_pairscore      0.925  0.167  0.283  [0.22, 0.34]     0.327    0.875  [0.85, 0.90]      0.723
  book_order        0.570  0.949  0.712  [0.67, 0.75]     0.591    0.855  [0.83, 0.88]      0.612
  cooccurrence      0.377  0.317  0.344  [0.29, 0.39]     0.101    0.549  [0.52, 0.58]      0.328
  cooc_x_order      0.744  0.297  0.424  [0.36, 0.48]    

In [ ]:
%cd /content/LearnKG
!python -m kg_extract.hybrid --help

/content/LearnKG
usage: hybrid.py [-h] --run RUN --out OUT [--source SOURCE] [--top-k TOP_K]

Hybrid knowledge-graph construction (no gold labels used anywhere).

Takes a finished `kg_extract.run` folder (grounded concepts + LLM relation scores) and the source text, then:
  1. scores EVERY directed concept pair u -> v ("u is a prerequisite of v") from text-only signals:
       order    forward-only gate: u's first mention precedes v's, scaled by the gap
       cooc     sentences mentioning both
       refd     RefD (Liang et al. 2015) over sentence sets
       llm_rel  the run's LLM relatedness (sum of its relation scores, either direction), if available
     score = forward_gate * mean of the percentile ranks of the available signals (fixed equal weights).
  2. keeps, for each concept, its top-k predecessors that are also *related* (cooc > 0 or LLM-related), so
     early foundational concepts do not become prerequisites of everything;
  3. adds the LLM's non-Prerequisite relations (P

In [ ]:
%cd /content/LearnKG

from pathlib import Path

for p in [
    Path("data_output/ck12_geometry_kgx_v2"),
    Path("data_output/ck12_geometry_kgx_v2_hybrid"),
    Path("data_input/ck12_geometry_cleaned/geometry.pairs"),
    Path("data_input/ck12_geometry_cleaned/geometry.preqs"),
]:
    print(f"{'FOUND' if p.exists() else 'MISSING'}: {p}")
    if p.is_dir():
        print("  Sample contents:", [x.name for x in list(p.iterdir())[:8]])

/content/LearnKG
FOUND: data_output/ck12_geometry_kgx_v2
  Sample contents: ['graph.html', 'relations.csv', 'run_meta.json', 'concept_candidates.csv', 'pair_scores.csv', 'concepts.csv', 'kg_eval']
FOUND: data_output/ck12_geometry_kgx_v2_hybrid
  Sample contents: ['relations.csv', 'run_meta.json', 'pair_scores.csv', 'concepts.csv', 'kg_eval']
FOUND: data_input/ck12_geometry_cleaned/geometry.pairs
FOUND: data_input/ck12_geometry_cleaned/geometry.preqs


In [ ]:
%cd /content/LearnKG

!python -m kg_extract.hybrid \
  --run data_output/ck12_geometry_kgx_v2 \
  --out data_output/ck12_geometry_kgx_v2_hybrid_k6 \
  --top-k 6

/content/LearnKG
modules.json: 100% 349/349 [00:00<00:00, 1.54MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 457kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 22.8MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 231kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.99MB/s]

model.safetensors: downloading bytes:  80% 72.3M/90.9M [00:00<00:00, 105MB/s, 5.96MB/s  ] 
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 73.0MB/s, 8.15MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 78.0MB/s, 8.84MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 7689.53it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 2.02MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 22.2MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 60.8MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 576kB/s]
config.json: 100% 190/190 [00:00<00:00, 940kB/s]
{
  "source_run": "data_output/ck12_geometry_kgx_v2",
  "uses_book_order": true,
  "si

In [ ]:
!python -m kg_eval.prereq_gold \
  --kg-dir data_output/ck12_geometry_kgx_v2_hybrid_k6

Gold concepts matched to KG: 65/84;  pairs matched: 989/1595;  located in text: 1431

MAIN (matched & located pairs)  (n=989, positives=293, prevalence=0.296)
  predictor             P      R     F1  F1 95% CI          MCC  ROC-AUC  AUC 95% CI       PR-AUC
  kg_prereq         0.790  0.334  0.470  [0.41, 0.53]     0.410    0.650  [0.62, 0.68]      0.492
  kg_closure        0.746  0.894  0.814  [0.78, 0.85]     0.731    0.898  [0.87, 0.92]      0.744
  kg_any_relation   0.758  0.341  0.471  [0.41, 0.52]     0.397    0.650  [0.62, 0.68]      0.487
  kg_undirected     0.384  0.345  0.363  [0.31, 0.41]     0.116    0.556  [0.53, 0.59]      0.327
  kg_pairscore      0.925  0.167  0.283  [0.22, 0.34]     0.327    0.875  [0.85, 0.90]      0.723
  book_order        0.570  0.949  0.712  [0.67, 0.75]     0.591    0.855  [0.83, 0.88]      0.612
  cooccurrence      0.377  0.317  0.344  [0.29, 0.39]     0.101    0.549  [0.52, 0.58]      0.328
  cooc_x_order      0.744  0.297  0.424  [0.36, 0.48]    

NEW TEST BY CHATGPT TOP K 8

In [ ]:
%cd /content/LearnKG

!python -m kg_extract.hybrid \
  --run data_output/ck12_geometry_kgx_v2 \
  --out data_output/ck12_geometry_kgx_v2_hybrid_k8 \
  --top-k 8

/content/LearnKG
Loading weights: 100% 103/103 [00:00<00:00, 6592.10it/s]
{
  "source_run": "data_output/ck12_geometry_kgx_v2",
  "uses_book_order": true,
  "signals": [
    "order",
    "cooc",
    "refd",
    "llm_rel"
  ],
  "top_k": 8,
  "prerequisite_edges": 1324,
  "llm_edges_kept": 24,
  "total_edges": 1348,
  "concepts": 204
}


In [ ]:
!python -m kg_eval.prereq_gold \
  --kg-dir data_output/ck12_geometry_kgx_v2_hybrid_k8

Gold concepts matched to KG: 65/84;  pairs matched: 989/1595;  located in text: 1431

MAIN (matched & located pairs)  (n=989, positives=293, prevalence=0.296)
  predictor             P      R     F1  F1 95% CI          MCC  ROC-AUC  AUC 95% CI       PR-AUC
  kg_prereq         0.776  0.379  0.509  [0.45, 0.56]     0.432    0.669  [0.64, 0.70]      0.514
  kg_closure        0.691  0.915  0.787  [0.75, 0.82]     0.694    0.895  [0.87, 0.92]      0.722
  kg_any_relation   0.762  0.382  0.509  [0.45, 0.56]     0.426    0.669  [0.64, 0.70]      0.511
  kg_undirected     0.379  0.389  0.384  [0.33, 0.43]     0.119    0.560  [0.53, 0.59]      0.328
  kg_pairscore      0.925  0.167  0.283  [0.22, 0.34]     0.327    0.875  [0.85, 0.90]      0.723
  book_order        0.570  0.949  0.712  [0.67, 0.75]     0.591    0.855  [0.83, 0.88]      0.612
  cooccurrence      0.377  0.317  0.344  [0.29, 0.39]     0.101    0.549  [0.52, 0.58]      0.328
  cooc_x_order      0.744  0.297  0.424  [0.36, 0.48]    

CONCEPT COVERAGE EXPERIMENTS

In [ ]:
%cd /content/LearnKG

import json
from pathlib import Path

report_path = Path(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/"
    "kg_eval/prereq_gold_report.json"
)

report = json.loads(report_path.read_text(encoding="utf-8"))

print("Top-level report keys:")
print(list(report.keys()))

print("\nReport preview:")
print(json.dumps(report, indent=2)[:12000])

/content/LearnKG
Top-level report keys:
['kg_dir', 'gold', 'gold_pairs', 'gold_positive', 'gold_concepts', 'gold_concepts_matched_to_kg', 'gold_concepts_low_confidence_match', 'gold_concepts_unmatched', 'pairs_matched', 'pairs_located_in_text', 'main', 'end_to_end', 'kg_prereq_weight_only', 'kg_prereq_edges_reversed_vs_gold', 'concept_matches', 'notes']

Report preview:
{
  "kg_dir": "data_output/ck12_geometry_kgx_v2_hybrid_k8",
  "gold": "AL-CPL (geometry.pairs / geometry.preqs)",
  "gold_pairs": 1595,
  "gold_positive": 496,
  "gold_concepts": 84,
  "gold_concepts_matched_to_kg": 65,
  "gold_concepts_low_confidence_match": [],
  "gold_concepts_unmatched": [
    "Collinearity",
    "Concentric",
    "Convex_polygon",
    "Coordinate_system",
    "Decagon",
    "Dodecagon",
    "Edge_(geometry)",
    "Face_(geometry)",
    "Geometry",
    "Heptagon",
    "Inverse_trigonometric_functions",
    "Nonagon",
    "Octagon",
    "Parallel_(geometry)",
    "Perpendicular",
    "Similarity_(geo

In [ ]:
%cd /content/LearnKG

import pandas as pd
import re
from difflib import SequenceMatcher

concept_path = (
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv"
)
concepts = pd.read_csv(concept_path)

print("Graph concept columns:", concepts.columns.tolist())
print("Number of graph concepts:", len(concepts))
display(concepts.head(5))

report_path = (
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/"
    "kg_eval/prereq_gold_report.json"
)
import json

with open(report_path, encoding="utf-8") as f:
    report = json.load(f)

unmatched = report["gold_concepts_unmatched"]

# Identify likely columns containing human-readable concept names.
name_cols = [
    c for c in concepts.columns
    if any(word in c.lower()
           for word in ["name", "label", "title", "concept", "term", "text"])
]

if not name_cols:
    print("No obvious name column found. Inspect the columns shown above.")
else:
    print("\nName columns used:", name_cols)

    def normalize(value):
        value = str(value).lower().replace("_", " ")
        value = re.sub(r"\([^)]*\)", "", value)
        return re.sub(r"[^a-z0-9]+", " ", value).strip()

    graph_names = []
    for _, row in concepts.iterrows():
        for col in name_cols:
            value = row[col]
            if pd.notna(value) and str(value).strip():
                graph_names.append((str(value), row.to_dict()))

    for gold in unmatched:
        target = normalize(gold)
        scored = []

        for name, row in graph_names:
            candidate = normalize(name)
            score = SequenceMatcher(None, target, candidate).ratio()

            # Also surface substring matches, even if their ratio is modest.
            if score >= 0.45 or (
                target and candidate and
                (target in candidate or candidate in target)
            ):
                scored.append((score, name, row))

        scored.sort(key=lambda x: x[0], reverse=True)

        print(f"\nGOLD: {gold}")
        if not scored:
            print("  No obvious name-level candidates.")
        else:
            for score, name, row in scored[:5]:
                print(f"  {score:.2f} | {name}")

/content/LearnKG
Graph concept columns: ['concept_id', 'concept_name', 'aliases', 'concept_type', 'definition', 'chapter', 'section', 'page', 'chunk_id', 'evidence']
Number of graph concepts: 204


,concept_id,concept_name,aliases,concept_type,definition,chapter,section,page,chunk_id,evidence
0,C0001,acute angle,NaN,Concept,Acute angles measure less than 90 degrees.,Chapter 1: Basics of Geometry,1.3 Rays and Angles,1,a6963ad59e6b4ecdb28d588d3d4ed83f,Acute angles measure less than 90 degrees.
1,C0002,acute triangle,NaN,Concept,An acute triangle has all angles less than 90 ...,Chapter 1: Basics of Geometry,1.5 Classifying Triangles,2,359afd06a29d48a2920c5a35ac7b1780,An acute triangle has all angles less than 90 ...
2,C0003,Adjacent angles,NaN,Concept,Adjacent angles share a vertex and a side but ...,Chapter 1: Basics of Geometry,1.4 Angle Pairs,1,7be8bb8f22d04966842fe3b83805e696,Adjacent angles share a vertex and a side but ...
3,C0004,Alternate exterior angles,NaN,Concept,Alternate exterior angles is outside the lines...,Chapter 3: Parallel and Perpendicular Lines,3.1 Lines and Angles,3,41c2869b034e4f70bc28b85b4f3a2424,Alternate exterior angles are outside the line...
4,C0005,alternate exterior angles theorem,NaN,Concept,The alternate exterior angles theorem states t...,Chapter 3: Parallel and Perpendicular Lines,3.2 Properties of Parallel Lines,3,4b0832ab2c974b7f98bf7062b01405cd,The alternate exterior angles theorem states t...



Name columns used: ['concept_id', 'concept_name', 'concept_type']

GOLD: Collinearity
  0.79 | Collinear points
  0.67 | corollary
  0.61 | linear pair
  0.60 | cylinder
  0.56 | Cosine Ratios

GOLD: Concentric
  0.71 | Concept
  0.71 | Concept
  0.71 | Concept
  0.71 | Concept
  0.71 | Concept

GOLD: Convex_polygon
  0.75 | Congruent polygons
  0.67 | polygon
  0.62 | regular polygon
  0.53 | Collinear points
  0.53 | Similar Polygons

GOLD: Coordinate_system
  0.50 | composite solid
  0.50 | congruent sides
  0.50 | corresponding parts
  0.50 | corresponding sides
  0.49 | Corresponding angles

GOLD: Decagon
  0.71 | hexagon
  0.67 | diagonal
  0.67 | pentagon
  0.62 | secant
  0.55 | cone

GOLD: Dodecagon
  0.62 | hexagon
  0.59 | diagonal
  0.59 | pentagon
  0.53 | secant
  0.50 | polygon

GOLD: Edge_(geometry)
  0.60 | degree
  0.50 | segments
  0.46 | rectangle

GOLD: Face_(geometry)
  0.57 | arc
  0.50 | cone
  0.50 | Distance
  0.50 | surface area

GOLD: Geometry
  0.70 | Trig

In [ ]:
%cd /content/LearnKG

import pandas as pd
import json
import re
from difflib import SequenceMatcher

concepts = pd.read_csv(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv"
).fillna("")

with open(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/"
    "kg_eval/prereq_gold_report.json",
    encoding="utf-8"
) as f:
    report = json.load(f)

unmatched = report["gold_concepts_unmatched"]

def norm(s):
    s = str(s).lower().replace("_", " ")
    s = re.sub(r"\([^)]*\)", "", s)
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

# Build searchable names using only the actual name and aliases.
search_rows = []
for _, row in concepts.iterrows():
    names = [row["concept_name"]]
    if row["aliases"]:
        names += re.split(r"[;,|]", str(row["aliases"]))

    for name in names:
        if name.strip():
            search_rows.append({
                "name": name.strip(),
                "concept_id": row["concept_id"],
                "definition": row["definition"],
                "chapter": row["chapter"],
                "section": row["section"],
                "evidence": row["evidence"],
            })

for gold in unmatched:
    target = norm(gold)
    scored = []

    for item in search_rows:
        candidate = norm(item["name"])
        score = SequenceMatcher(None, target, candidate).ratio()

        if target == candidate:
            score = 1.0

        if score >= 0.40:
            scored.append((score, item))

    scored.sort(key=lambda x: x[0], reverse=True)

    print(f"\n{'='*75}\nGOLD CONCEPT: {gold}")
    if not scored:
        print("No close name/alias candidates.")
        continue

    for score, item in scored[:3]:
        print(
            f"\n  Similarity: {score:.2f}"
            f"\n  Candidate:  {item['name']} ({item['concept_id']})"
            f"\n  Definition: {item['definition']}"
            f"\n  Location:   {item['chapter']} / {item['section']}"
            f"\n  Evidence:   {item['evidence'][:350]}"
        )

/content/LearnKG

GOLD CONCEPT: Collinearity

  Similarity: 0.79
  Candidate:  Collinear points (C0036)
  Definition: Collinear points are points that lie on the same line.
  Location:   Chapter 1: Basics of Geometry / 1.1 Points, Lines, and Planes
  Evidence:   Collinear points are points that lie on the same line.

  Similarity: 0.67
  Candidate:  corollary (C0054)
  Definition: The corollary states that each angle of an equilateral triangle measures 60 degrees.
  Location:   Chapter 4: Triangles and Congruence / 4.1 Triangle Sums
  Evidence:   The corollary states that each angle of an equilateral triangle measures 60 degrees.

  Similarity: 0.61
  Candidate:  linear pair (C0104)
  Definition: A linear pair is two adjacent angles that form a straight line.
  Location:   Chapter 1: Basics of Geometry / 1.4 Angle Pairs
  Evidence:   A linear pair is two adjacent angles that form a straight line.

GOLD CONCEPT: Concentric

  Similarity: 0.67
  Candidate:  centroid (C0029)
  Definition:

In [ ]:
%cd /content/LearnKG

from pathlib import Path
import pandas as pd
import re

targets = [
    "Collinearity", "Concentric", "Convex polygon",
    "Coordinate system", "Decagon", "Dodecagon",
    "Edge", "Face", "Geometry", "Heptagon",
    "Inverse trigonometric functions", "Nonagon",
    "Octagon", "Parallel", "Perpendicular",
    "Similarity", "Sine", "Skew lines", "Tessellation"
]

def variants(term):
    base = term.replace("_", " ")
    base = re.sub(r"\([^)]*\)", "", base).strip()
    return list(dict.fromkeys([
        base, base.lower(), base.title(),
        base.replace(" ", ""), base.replace(" ", "-")
    ]))

# Search text-like files in the original extraction run and input folder.
roots = [
    Path("data_output/ck12_geometry_kgx_v2"),
    Path("data_input")
]

files_to_search = []
for root in roots:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() in {
                ".txt", ".md", ".csv", ".json", ".jsonl"
            }:
                # Skip generated evaluation outputs and large graph HTML files.
                if "kg_eval" not in p.parts and p.stat().st_size < 30_000_000:
                    files_to_search.append(p)

print("Text-like files found:", len(files_to_search))

for term in targets:
    patterns = [re.compile(re.escape(v), re.I) for v in variants(term)]
    hits = []

    for path in files_to_search:
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue

        for pattern in patterns:
            match = pattern.search(text)
            if match:
                start = max(0, match.start() - 100)
                end = min(len(text), match.end() + 160)
                snippet = re.sub(r"\s+", " ", text[start:end])
                hits.append((str(path), snippet))
                break

    print(f"\n{term}: {'FOUND' if hits else 'NOT FOUND in searched files'}")
    for path, snippet in hits[:2]:
        print(" ", path)
        print("   ", snippet)

In [ ]:
%cd /content/LearnKG

from pathlib import Path
import re

textbook = Path("data_input/ck12_geometry/ck12_basic_geometry_concepts.txt")
assert textbook.exists(), f"Textbook not found: {textbook}"

text = textbook.read_text(encoding="utf-8", errors="ignore")
print(f"Textbook loaded: {len(text):,} characters")
print(f"Word count: {len(text.split()):,}")

targets = [
    "Collinearity",
    "Concentric",
    "Convex polygon",
    "Coordinate system",
    "Decagon",
    "Dodecagon",
    "Edge",
    "Face",
    "Geometry",
    "Heptagon",
    "Inverse trigonometric functions",
    "Nonagon",
    "Octagon",
    "Parallel",
    "Perpendicular",
    "Similarity",
    "Sine",
    "Skew lines",
    "Tessellation",
]

def make_patterns(term):
    variants = {
        term,
        term.replace("_", " "),
        term.replace("_", " ").replace("(geometry)", "").strip(),
    }
    patterns = []
    for variant in variants:
        words = variant.split()
        if not words:
            continue
        # Allow spaces, hyphens, or underscores between words.
        pattern = r"(?<!\w)" + r"[\s_-]+".join(
            re.escape(word) for word in words
        ) + r"(?!\w)"
        patterns.append(re.compile(pattern, re.IGNORECASE))
    return patterns

for term in targets:
    patterns = make_patterns(term)
    matches = []

    for pattern in patterns:
        for match in pattern.finditer(text):
            if not any(m.start() == match.start() for m in matches):
                matches.append(match)

    matches.sort(key=lambda m: m.start())

    print(f"\n{'=' * 65}")
    print(f"{term}: {len(matches)} occurrence(s)")

    for match in matches[:3]:
        start = max(0, match.start() - 120)
        end = min(len(text), match.end() + 180)
        snippet = re.sub(r"\s+", " ", text[start:end])
        print(f"  Position {match.start()}: ...{snippet}...")

/content/LearnKG
Textbook loaded: 20,827 characters
Word count: 3,509

Collinearity: 1 occurrence(s)
  Position 20722: ...n of a reflection and a translation parallel to the line of reflection. Isometries preserve angle measure, betweenness, collinearity, and distance. Any rigid transformation can be expressed as the composition of reflections. ...

Concentric: 0 occurrence(s)

Convex polygon: 1 occurrence(s)
  Position 11004: ...Polygons The interior angle sum of a polygon with n sides is n minus 2 times 180 degrees. The exterior angle sum of any convex polygon is always 360 degrees. A diagonal is a segment connecting two non-adjacent vertices. 6.2 Properties of Parallelograms A parallelogram is a quadrilateral with both pairs of opposi...

Coordinate system: 0 occurrence(s)

Decagon: 0 occurrence(s)

Dodecagon: 0 occurrence(s)

Edge: 0 occurrence(s)

Face: 0 occurrence(s)

Geometry: 3 occurrence(s)
  Position 12: ...CK-12 Basic Geometry Concepts Chapter 1: Basics of Geometry 1.1 Poin

In [ ]:
%cd /content/LearnKG

import pandas as pd
import re
from pathlib import Path

text_path = Path("data_input/ck12_geometry/ck12_basic_geometry_concepts.txt")
kg_path = Path("data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv")

text = text_path.read_text(encoding="utf-8", errors="ignore")
concepts = pd.read_csv(kg_path).fillna("")

targets = [
    "Collinearity", "Concentric", "Convex polygon",
    "Coordinate system", "Decagon", "Dodecagon",
    "Edge", "Face", "Geometry", "Heptagon",
    "Inverse trigonometric functions", "Nonagon",
    "Octagon", "Parallel", "Perpendicular",
    "Similarity", "Sine", "Skew lines", "Tessellation"
]

def normalize(s):
    s = str(s).lower().replace("_", " ")
    s = re.sub(r"\([^)]*\)", "", s)
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

def find_text_occurrences(term):
    # Exact phrase matching, allowing spaces, hyphens, or underscores.
    words = normalize(term).split()
    if not words:
        return []
    pattern = re.compile(
        r"(?<!\w)" + r"[\s_-]+".join(map(re.escape, words)) + r"(?!\w)",
        re.I
    )
    return list(pattern.finditer(text))

for target in targets:
    target_norm = normalize(target)
    target_words = set(target_norm.split())
    occurrences = find_text_occurrences(target)

    print(f"\n{'=' * 75}")
    print(f"{target} | textbook exact-phrase hits: {len(occurrences)}")

    # Search names and aliases first; do not use definitions to claim a match.
    candidates = []
    for _, row in concepts.iterrows():
        name = str(row["concept_name"])
        aliases = str(row["aliases"])
        labels = [name] + re.split(r"[;,|]", aliases)

        best_overlap = 0
        exact = False

        for label in labels:
            label_norm = normalize(label)
            if not label_norm:
                continue

            if label_norm == target_norm:
                exact = True

            label_words = set(label_norm.split())
            overlap = len(target_words & label_words) / max(
                1, len(target_words | label_words)
            )
            best_overlap = max(best_overlap, overlap)

        if exact or best_overlap >= 0.25:
            candidates.append((exact, best_overlap, row))

    candidates.sort(key=lambda x: (not x[0], -x[1]))

    if not candidates:
        print("  No plausible name/alias candidate found in the KG.")
    else:
        for exact, score, row in candidates[:5]:
            print(
                f"  - {row['concept_name']} "
                f"[ID: {row['concept_id']}; "
                f"{'EXACT LABEL' if exact else f'word overlap={score:.2f}'}]"
            )
            print(f"    Aliases: {row['aliases']}")
            print(f"    Definition: {row['definition']}")
            print(f"    Location: chapter={row['chapter']}, "
                  f"section={row['section']}, page={row['page']}")

    # Show one source passage for terms that do occur in the textbook.
    if occurrences:
        match = occurrences[0]
        start = max(0, match.start() - 160)
        end = min(len(text), match.end() + 240)
        snippet = re.sub(r"\s+", " ", text[start:end])
        print(f"  Source context: ...{snippet}...")

/content/LearnKG

Collinearity | textbook exact-phrase hits: 1
  No plausible name/alias candidate found in the KG.
  Source context: ...ons A glide reflection is the composition of a reflection and a translation parallel to the line of reflection. Isometries preserve angle measure, betweenness, collinearity, and distance. Any rigid transformation can be expressed as the composition of reflections. ...

Concentric | textbook exact-phrase hits: 0
  No plausible name/alias candidate found in the KG.

Convex polygon | textbook exact-phrase hits: 1
  - polygon [ID: C0135; word overlap=0.50]
    Aliases: Polygons
    Definition: A polygon is a closed plane figure with three or more straight sides.
    Location: chapter=Chapter 1: Basics of Geometry, section=1.6 Classifying Polygons, page=2
  - regular polygon [ID: C0152; word overlap=0.33]
    Aliases: 
    Definition: A regular polygon has all sides congruent and all angles congruent.
    Location: chapter=Chapter 1: Basics of Geometry, se

In [ ]:
%cd /content/LearnKG

from pathlib import Path
import json

# 1. Inspect the gold benchmark files.
for path in [
    Path("data_input/ck12_geometry_cleaned/geometry.pairs"),
    Path("data_input/ck12_geometry_cleaned/geometry.preqs"),
]:
    print(f"\n{'=' * 70}\nFILE: {path}")
    if path.exists():
        with path.open(encoding="utf-8", errors="replace") as f:
            for _, line in zip(range(5), f):
                print(line.rstrip())
    else:
        print("Not found")

# 2. Inspect how the evaluator performs concept matching.
evaluator = Path("kg_eval/prereq_gold.py")
print(f"\n{'=' * 70}\nEVALUATOR: {evaluator}")

if evaluator.exists():
    lines = evaluator.read_text(encoding="utf-8").splitlines()
    keywords = ("match", "alias", "normalize", "unmatched", "fuzzy")
    relevant = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    shown = set()
    for i in relevant:
        for j in range(max(0, i - 3), min(len(lines), i + 5)):
            if j not in shown:
                print(f"{j + 1:4}: {lines[j]}")
                shown.add(j)
        print()
else:
    print("Evaluator source file not found.")

# 3. Show the report's existing mapping for unmatched concepts.
report_path = Path(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/"
    "kg_eval/prereq_gold_report.json"
)

print(f"\n{'=' * 70}\nREPORT MAPPINGS")
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Report keys:", list(report.keys()))
    print("Unmatched concepts:", report.get("gold_concepts_unmatched"))
    print("Low-confidence matches:",
          report.get("gold_concepts_low_confidence_match"))
else:
    print("Report not found.")

/content/LearnKG

FILE: data_input/ck12_geometry_cleaned/geometry.pairs
Parallelogram,Rotational_symmetry
Angle,Isosceles_triangle
Angle,Pythagorean_theorem
Edge_(geometry),Acute_and_obtuse_triangles
Polygon,Line_segment

FILE: data_input/ck12_geometry_cleaned/geometry.preqs
Congruence_(geometry),Line_segment
Bisection,Point_(geometry)
Arc_(geometry),Circle
Edge_(geometry),Line_(geometry)
Diameter,Point_(geometry)

EVALUATOR: kg_eval/prereq_gold.py
  21:                    |S_A & S_B|/|S_A| - |S_A & S_B|/|S_B|
  22: 
  23: Evaluation sets:
  24:   main        both gold concepts matched to distinct KG concepts AND located in the text
  25:               (all predictors comparable here)
  26:   end_to_end  all gold pairs; unmatched concepts count as "no edge" (KG predictors only)
  27: 
  28: Run from LearnKG/:

  29:   python -m kg_eval.prereq_gold [--kg-dir data_output/ck12_geometry] [--out DIR]
  30: """

  39: from evaluate_prerequisite_extraction import (
  40:     GEOMETRY_SYNONYM_

In [ ]:
%cd /content/LearnKG

from pathlib import Path
import ast

path = Path("evaluate_prerequisite_extraction.py")
assert path.exists(), f"Could not find {path}"

source = path.read_text(encoding="utf-8")
lines = source.splitlines()

# Print the synonym map and the matching-related functions.
tree = ast.parse(source)

wanted = {
    "GEOMETRY_SYNONYM_MAP",
    "normalize_gold_concept_name",
    "build_extracted_concept_index",
    "match_gold_concept",
}

print("File:", path)
print("Total lines:", len(lines))

found = set()

for node in tree.body:
    name = getattr(node, "name", None)

    if name in wanted:
        found.add(name)
        start = node.lineno - 1
        end = node.end_lineno
        print(f"\n{'=' * 75}")
        print(f"{name} (lines {start + 1}-{end})")
        print("\n".join(
            f"{i + 1:4}: {lines[i]}"
            for i in range(start, end)
        ))

print("\nMissing definitions:", sorted(wanted - found))

/content/LearnKG
File: evaluate_prerequisite_extraction.py
Total lines: 480

normalize_gold_concept_name (lines 21-31)
  21: def normalize_gold_concept_name(raw_name: str) -> str:
  22:     """
  23:     Normalizes AL-CPL / Wikipedia concept titles:
  24:     - Replaces underscores with spaces
  25:     - Removes parenthetical disambiguation e.g. '(geometry)', '(mathematics)'
  26:     - Applies ConceptNormalizer.clean_name
  27:     """
  28:     s = raw_name.replace("_", " ")
  29:     s = re.sub(r"\s*\([^)]*\)", "", s).strip()
  30:     s = ConceptNormalizer.clean_name(s)
  31:     return s

build_extracted_concept_index (lines 57-96)
  57: def build_extracted_concept_index(concepts_df: pd.DataFrame) -> Tuple[Dict[str, Dict[str, Any]], Dict[str, str], ConceptNormalizer]:
  58:     """
  59:     Builds lookup tables for extracted concepts.
  60:     Returns:
  61:       - concepts_by_id: {cid: {name, aliases, ...}}
  62:       - term_to_cid: {lowercased_term: cid}
  63:       - norma

In [ ]:
%cd /content/LearnKG

import json
import ast
import pandas as pd
from pathlib import Path

source_path = Path("evaluate_prerequisite_extraction.py")
source = source_path.read_text(encoding="utf-8")
tree = ast.parse(source)

# Locate and print the actual synonym-map assignment.
print("=" * 75)
print("GEOMETRY_SYNONYM_MAP")

found_map = False
for node in ast.walk(tree):
    if isinstance(node, (ast.Assign, ast.AnnAssign)):
        targets = node.targets if isinstance(node, ast.Assign) else [node.target]
        if any(
            isinstance(t, ast.Name) and t.id == "GEOMETRY_SYNONYM_MAP"
            for t in targets
        ):
            print(ast.get_source_segment(source, node))
            found_map = True
            break

if not found_map:
    print("Map not defined directly in this file; inspect its imports.")

# Read existing evaluation results.
report_path = Path(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/"
    "kg_eval/prereq_gold_report.json"
)
report = json.loads(report_path.read_text(encoding="utf-8"))

targets = report["gold_concepts_unmatched"]
matches = report["concept_matches"]

print("\n" + "=" * 75)
print("CURRENT MATCH DECISIONS")
for gold in targets:
    print(f"{gold:40} -> {matches.get(gold)}")

# Inspect candidate names and aliases for potentially relevant concepts.
concepts_path = Path(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv"
)
concepts = pd.read_csv(concepts_path).fillna("")

print("\n" + "=" * 75)
print("RELATED EXTRACTED CONCEPTS")
keywords = [
    "collinear", "convex", "inverse trigonometric",
    "parallel", "perpendicular", "similarity",
    "sine", "geometry", "polygon"
]

for keyword in keywords:
    rows = concepts[
        concepts["concept_name"].str.contains(keyword, case=False, regex=False)
        | concepts["aliases"].str.contains(keyword, case=False, regex=False)
    ]
    print(f"\n[{keyword}]")
    if rows.empty:
        print("  No matching name or alias")
    else:
        for _, row in rows.iterrows():
            print(f"  {row['concept_id']}: {row['concept_name']}"
                  f" | aliases={row['aliases']}")

/content/LearnKG
GEOMETRY_SYNONYM_MAP
GEOMETRY_SYNONYM_MAP: Dict[str, List[str]] = {
    # Direct AL-CPL → CK-12 name mappings
    "Acute and obtuse triangles": ["Acute Triangle", "Obtuse Triangle"],
    "Bisection":                  ["Perpendicular Bisector", "Angle Bisector", "Bisector"],
    "Cartesian coordinate system":["Coordinate Plane", "Coordinate System", "Cartesian Coordinate"],
    "Coordinate system":          ["Coordinate Plane", "Coordinate System"],
    "Cross section":              ["Cross Section"],
    "Grade":                      ["Slope"],
    "Hendecagon":                  ["Hendecagon", "11-gon"],
    "Internal and external angle": ["Interior Angle", "Exterior Angle"],
    "Reflection symmetry":        ["Reflection Symmetry", "Line Symmetry", "Symmetry"],
    "Scaling":                    ["Dilation", "Scale Factor"],
    "Secant line":                ["Secant"],
    "Self-similarity":            ["Self Similarity", "Fractal"],
    "Skew lines":                 

In [ ]:
%cd /content/LearnKG

import pandas as pd

concepts = pd.read_csv(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv"
).fillna("")

mask = (
    concepts["concept_name"].str.contains(
        r"coordinate|plane|cartesian", case=False, regex=True
    )
    | concepts["aliases"].str.contains(
        r"coordinate|plane|cartesian", case=False, regex=True
    )
)

print(concepts.loc[
    mask, ["concept_id", "concept_name", "aliases", "definition"]
].to_string(index=False))

/content/LearnKG
concept_id concept_name aliases                                                                           definition
     C0132        plane  Planes A plane is a flat two-dimensional surface that extends infinitely in all directions.


In [ ]:
%cd /content/LearnKG

!python -m kg_eval.prereq_gold \
  --kg-dir data_output/ck12_geometry_kgx_v2_hybrid_k8

/content/LearnKG
Gold concepts matched to KG: 65/84;  pairs matched: 989/1595;  located in text: 1431

MAIN (matched & located pairs)  (n=989, positives=293, prevalence=0.296)
  predictor             P      R     F1  F1 95% CI          MCC  ROC-AUC  AUC 95% CI       PR-AUC
  kg_prereq         0.776  0.379  0.509  [0.45, 0.56]     0.432    0.669  [0.64, 0.70]      0.514
  kg_closure        0.691  0.915  0.787  [0.75, 0.82]     0.694    0.895  [0.87, 0.92]      0.722
  kg_any_relation   0.762  0.382  0.509  [0.45, 0.56]     0.426    0.669  [0.64, 0.70]      0.511
  kg_undirected     0.379  0.389  0.384  [0.33, 0.43]     0.119    0.560  [0.53, 0.59]      0.328
  kg_pairscore      0.925  0.167  0.283  [0.22, 0.34]     0.327    0.875  [0.85, 0.90]      0.723
  book_order        0.570  0.949  0.712  [0.67, 0.75]     0.591    0.855  [0.83, 0.88]      0.612
  cooccurrence      0.377  0.317  0.344  [0.29, 0.39]     0.101    0.549  [0.52, 0.58]      0.328
  cooc_x_order      0.744  0.297  0.424 

In [ ]:
import json
from pathlib import Path

path = Path(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/"
    "kg_eval/prereq_gold_report.json"
)
report = json.loads(path.read_text(encoding="utf-8"))

print("Matched concepts:",
      report["gold_concepts_matched_to_kg"], "/", report["gold_concepts"])
print("Unmatched concepts:")
for concept in report["gold_concepts_unmatched"]:
    print(" -", concept)

print("\nMapping verification:")
for gold in ["Collinearity", "Sine", "Coordinate_system"]:
    print(gold, "->", report["concept_matches"].get(gold))

Matched concepts: 65 / 84
Unmatched concepts:
 - Collinearity
 - Concentric
 - Convex_polygon
 - Coordinate_system
 - Decagon
 - Dodecagon
 - Edge_(geometry)
 - Face_(geometry)
 - Geometry
 - Heptagon
 - Inverse_trigonometric_functions
 - Nonagon
 - Octagon
 - Parallel_(geometry)
 - Perpendicular
 - Similarity_(geometry)
 - Sine
 - Skew_lines
 - Tessellation

Mapping verification:
Collinearity -> {'cid': None, 'kg_name': None, 'confidence': 'unmatched'}
Sine -> {'cid': None, 'kg_name': None, 'confidence': 'unmatched'}
Coordinate_system -> {'cid': None, 'kg_name': None, 'confidence': 'unmatched'}


In [ ]:
%cd /content/LearnKG

import importlib
import inspect
import evaluate_prerequisite_extraction as ev

# Reload the module so Python reads the current saved file
ev = importlib.reload(ev)

print("=" * 70)
print("1. ACTUAL MODULE LOCATION")
print(inspect.getfile(ev))

print("\n" + "=" * 70)
print("2. LIVE SYNONYM MAP ENTRIES")
for key in ["Collinearity", "Sine", "Coordinate system"]:
    print(f"{key!r}: {ev.GEOMETRY_SYNONYM_MAP.get(key)}")

print("\n" + "=" * 70)
print("3. MATCHER FUNCTION SOURCE")
print(inspect.getsource(ev.match_gold_concept))

print("\n" + "=" * 70)
print("4. DIRECT MATCH TESTS")

import pandas as pd

concepts = pd.read_csv(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv"
).fillna("")

index = ev.build_extracted_concept_index(concepts)

for gold in ["Collinearity", "Sine", "Coordinate_system"]:
    try:
        result = ev.match_gold_concept(gold, index)
        print(f"{gold!r} -> {result}")
    except Exception as exc:
        print(f"{gold!r} -> ERROR: {type(exc).__name__}: {exc}")

/content/LearnKG
1. ACTUAL MODULE LOCATION
/content/LearnKG/evaluate_prerequisite_extraction.py

2. LIVE SYNONYM MAP ENTRIES
'Collinearity': None
'Sine': None
'Coordinate system': ['Coordinate Plane', 'Coordinate System']

3. MATCHER FUNCTION SOURCE
def match_gold_concept(
    gold_raw: str,
    term_to_cid: Dict[str, str],
    concepts_by_id: Dict[str, Dict[str, Any]],
    normalizer: ConceptNormalizer,
    fuzzy_threshold: float = 0.82,  # Slightly relaxed to catch more textbook variants
) -> Tuple[Optional[str], str, float, List[str]]:
    """
    Matches an AL-CPL gold concept name to an extracted concept.
    First checks the GEOMETRY_SYNONYM_MAP for known CK-12 name equivalents,
    then falls back through exact, plural, token, and fuzzy matching.
    Returns:
      (matched_cid, confidence_level, score, candidate_notes)
      confidence_level: 'high', 'medium', 'low', or 'unmatched'
    """
    normalized_gold = normalize_gold_concept_name(gold_raw)
    gold_lower = normalized_g

In [ ]:
%cd /content/LearnKG

import importlib
import inspect
import pandas as pd
import evaluate_prerequisite_extraction as ev

ev = importlib.reload(ev)

concepts = pd.read_csv(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv"
).fillna("")

# Build the three structures required by the actual matcher.
term_to_cid, concepts_by_id = ev.build_extracted_concept_index(concepts)

# Inspect the index builder to identify how its normalizer is initialized.
print("INDEX BUILDER:")
print(inspect.getsource(ev.build_extracted_concept_index))

print("\nNORMALIZER CLASS:")
print(inspect.getsource(ev.ConceptNormalizer))

print("\nCURRENT SYNONYM MAPPINGS:")
for key in ["Collinearity", "Sine", "Coordinate system"]:
    print(f"{key!r}: {ev.GEOMETRY_SYNONYM_MAP.get(key)}")

print("\nINDEX CHECK:")
for term in ["collinear points", "sine ratios", "coordinate plane", "plane"]:
    print(f"{term!r}: {term in term_to_cid}")

/content/LearnKG


ValueError: too many values to unpack (expected 2)

In [ ]:
%cd /content/LearnKG

import inspect
import evaluate_prerequisite_extraction as ev

print("FUNCTION SOURCE:")
print(inspect.getsource(ev.build_extracted_concept_index))

print("\nRETURN STRUCTURE:")
result = ev.build_extracted_concept_index(
    __import__("pandas").read_csv(
        "data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv"
    ).fillna("")
)

print("Return type:", type(result).__name__)
print("Number of returned values:", len(result))

for i, item in enumerate(result):
    print(f"\nItem {i}: type={type(item).__name__}")
    if isinstance(item, dict):
        print("  Dictionary size:", len(item))
        print("  Sample keys:", list(item.keys())[:5])
    else:
        print("  Sample:", repr(item)[:300])

print("\nNORMALIZER CONSTRUCTOR:")
print(inspect.signature(ev.ConceptNormalizer))

/content/LearnKG
FUNCTION SOURCE:
def build_extracted_concept_index(concepts_df: pd.DataFrame) -> Tuple[Dict[str, Dict[str, Any]], Dict[str, str], ConceptNormalizer]:
    """
    Builds lookup tables for extracted concepts.
    Returns:
      - concepts_by_id: {cid: {name, aliases, ...}}
      - term_to_cid: {lowercased_term: cid}
      - normalizer: ConceptNormalizer instance for helper matching methods
    """
    normalizer = ConceptNormalizer()
    concepts_by_id: Dict[str, Dict[str, Any]] = {}
    term_to_cid: Dict[str, str] = {}

    for _, row in concepts_df.iterrows():
        cid = str(row["concept_id"]).strip()
        cname = str(row["concept_name"]).strip()
        aliases_raw = str(row.get("aliases", ""))
        aliases = [a.strip() for a in aliases_raw.split(",") if a.strip()] if aliases_raw and aliases_raw != "nan" else []
        
        concepts_by_id[cid] = {
            "concept_id": cid,
            "concept_name": cname,
            "aliases": aliases,
          

In [ ]:
%cd /content/LearnKG

import importlib
import pandas as pd
import evaluate_prerequisite_extraction as ev

ev = importlib.reload(ev)

concepts = pd.read_csv(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv"
).fillna("")

# Correct return order: 3 values
concepts_by_id, term_to_cid, normalizer = (
    ev.build_extracted_concept_index(concepts)
)

print("=== 1. CURRENT SYNONYM MAP ===")
for key in [
    "Collinearity",
    "Sine",
    "Coordinate system",
]:
    print(f"{key!r}: {ev.GEOMETRY_SYNONYM_MAP.get(key)}")

print("\n=== 2. EXTRACTED TERM INDEX ===")
for term in [
    "collinear points",
    "sine ratios",
    "coordinate plane",
    "plane",
]:
    cid = term_to_cid.get(ev.ConceptNormalizer.clean_name(term).lower())
    details = concepts_by_id.get(cid, {}) if cid else {}
    print(
        f"{term!r}: "
        f"cid={cid}, "
        f"concept={details.get('concept_name')}"
    )

print("\n=== 3. ACTUAL MATCHER RESULTS ===")
for gold in [
    "Collinearity",
    "Sine",
    "Coordinate system",
]:
    result = ev.match_gold_concept(
        gold,
        term_to_cid,
        concepts_by_id,
        normalizer,
    )
    print(f"{gold!r} -> {result}")

/content/LearnKG
=== 1. CURRENT SYNONYM MAP ===
'Collinearity': None
'Sine': None
'Coordinate system': ['Coordinate Plane', 'Coordinate System']

=== 2. EXTRACTED TERM INDEX ===
'collinear points': cid=C0036, concept=Collinear points
'sine ratios': cid=C0171, concept=Sine Ratios
'coordinate plane': cid=None, concept=None
'plane': cid=C0132, concept=plane

=== 3. ACTUAL MATCHER RESULTS ===
'Collinearity' -> (None, 'unmatched', 0.0, ['No candidate match above threshold'])
'Sine' -> (None, 'unmatched', 0.0, ['No candidate match above threshold'])
'Coordinate system' -> (None, 'unmatched', 0.0, ['No candidate match above threshold'])


In [ ]:

%cd /content/LearnKG

import csv
import re
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter

import pandas as pd
import evaluate_prerequisite_extraction as ev

# ---------- 1. Load the existing artifacts ----------
base = Path("data_output/ck12_geometry_kgx_v2_hybrid_k8")
source_path = Path(
    "data_input/ck12_geometry/ck12_basic_geometry_concepts.txt"
)
benchmark_dir = Path("data_input/ck12_geometry_cleaned")

source_text = source_path.read_text(encoding="utf-8", errors="replace")
concepts_df = pd.read_csv(base / "concepts.csv").fillna("")

concepts_by_id, term_to_cid, normalizer = (
    ev.build_extracted_concept_index(concepts_df)
)

# ---------- 2. Reconstruct gold concepts from the cleaned benchmark ----------
gold_concepts = set()

for filename in ["geometry.pairs", "geometry.preqs"]:
    path = benchmark_dir / filename

    if not path.exists():
        raise FileNotFoundError(f"Missing benchmark file: {path}")

    with path.open(encoding="utf-8-sig", newline="") as f:
        for row in csv.reader(f):
            row = [cell.strip() for cell in row if cell.strip()]

            if len(row) < 2:
                continue

            # Skip a possible header row.
            if any(
                cell.lower() in {
                    "source", "target", "concept1", "concept2",
                    "prerequisite", "dependent"
                }
                for cell in row[:2]
            ):
                continue

            gold_concepts.update(row[:2])

gold_concepts = sorted(gold_concepts)

# ---------- 3. Prepare conservative phrase matching ----------
def phrase_normalize(value):
    value = ev.normalize_gold_concept_name(str(value))
    value = value.lower()
    return re.sub(r"[^a-z0-9]+", " ", value).strip()

normalized_source = " " + phrase_normalize(source_text) + " "

def occurs_in_source(term):
    term_norm = phrase_normalize(term)
    if not term_norm:
        return False
    return f" {term_norm} " in normalized_source

# Canonical names and aliases for candidate inspection.
candidate_terms = []

for cid, info in concepts_by_id.items():
    name = info.get("concept_name", "")
    if name:
        candidate_terms.append((name, cid))

    for alias in info.get("aliases", []):
        if alias:
            candidate_terms.append((alias, cid))

# ---------- 4. Audit every gold concept ----------
rows = []

for gold in gold_concepts:
    cid, method, score, notes = ev.match_gold_concept(
        gold,
        term_to_cid,
        concepts_by_id,
        normalizer,
    )

    matched_name = (
        concepts_by_id[cid]["concept_name"] if cid else ""
    )

    # Rank candidates for diagnosis only; these are NOT automatic matches.
    gold_norm = phrase_normalize(gold)
    candidates = []

    for term, candidate_cid in candidate_terms:
        term_norm = phrase_normalize(term)
        if not term_norm:
            continue

        similarity = SequenceMatcher(
            None, gold_norm, term_norm
        ).ratio()

        if similarity >= 0.35:
            candidates.append(
                (similarity, term, candidate_cid)
            )

    candidates.sort(reverse=True)
    top_candidates = []
    seen = set()

    for similarity, term, candidate_cid in candidates:
        key = (term, candidate_cid)
        if key in seen:
            continue
        seen.add(key)
        top_candidates.append(
            f"{term} [{candidate_cid}; {similarity:.2f}]"
        )
        if len(top_candidates) == 3:
            break

    source_hit = occurs_in_source(gold)

    rows.append({
        "gold_concept": gold,
        "source_phrase_found": source_hit,
        "graph_match": matched_name or "UNMATCHED",
        "concept_id": cid or "",
        "match_method": method,
        "match_score": score,
        "top_graph_candidates": " | ".join(top_candidates),
    })

audit = pd.DataFrame(rows)

# ---------- 5. Summarize coverage ----------
matched = audit["concept_id"].ne("")
source_found = audit["source_phrase_found"]

print("=" * 72)
print("BENCHMARK / TEXTBOOK / GRAPH AUDIT")
print("=" * 72)
print("Gold concepts reconstructed:", len(audit))
print("Graph nodes:", len(concepts_df))
print("Gold concepts matched to graph:", int(matched.sum()))
print("Gold concepts unmatched:", int((~matched).sum()))
print("Gold labels found as normalized phrases in source:",
      int(source_found.sum()))
print("Gold labels NOT found as normalized phrases in source:",
      int((~source_found).sum()))

print("\nNOTE: phrase presence is only a first-pass test.")
print("It does not establish that the concept is explained sufficiently")
print("for extraction, and phrase absence does not prove the idea is absent.")

print("\n" + "=" * 72)
print("ALL GOLD CONCEPTS — SOURCE AND GRAPH STATUS")
print("=" * 72)

for _, r in audit.iterrows():
    print(
        f"\n{r['gold_concept']}"
        f"\n  Source phrase found: {r['source_phrase_found']}"
        f"\n  Graph match: {r['graph_match']}"
        f"\n  Match method: {r['match_method']}"
    )

    if r["graph_match"] == "UNMATCHED":
        print(f"  Possible graph candidates: "
              f"{r['top_graph_candidates'] or 'None'}")

print("\n" + "=" * 72)
print("UNMATCHED CONCEPTS — PRIORITY REVIEW")
print("=" * 72)

unmatched = audit.loc[~matched].copy()

for _, r in unmatched.iterrows():
    if not r["source_phrase_found"]:
        initial_class = "NO EXACT NORMALIZED PHRASE IN SOURCE"
    else:
        initial_class = "PHRASE FOUND, BUT NO GRAPH MATCH"

    print(f"\n[{initial_class}] {r['gold_concept']}")
    print("  Candidates:", r["top_graph_candidates"] or "None")

# Keep the full audit available for follow-up analysis.
audit.to_csv("/content/learnkg_gold_concept_audit.csv", index=False)
print("\nFull audit saved to /content/learnkg_gold_concept_audit.csv")

if len(audit) != 84:
    print(
        "\nWARNING: Expected 84 unique gold concepts, but reconstructed",
        len(audit),
        ". Check the benchmark file format/header before interpreting results."
    )


/content/LearnKG
BENCHMARK / TEXTBOOK / GRAPH AUDIT
Gold concepts reconstructed: 84
Graph nodes: 204
Gold concepts matched to graph: 65
Gold concepts unmatched: 19
Gold labels found as normalized phrases in source: 65
Gold labels NOT found as normalized phrases in source: 19

NOTE: phrase presence is only a first-pass test.
It does not establish that the concept is explained sufficiently
for extraction, and phrase absence does not prove the idea is absent.

ALL GOLD CONCEPTS — SOURCE AND GRAPH STATUS

Acute_and_obtuse_triangles
  Source phrase found: False
  Graph match: acute triangle
  Match method: high

Altitude_(triangle)
  Source phrase found: True
  Graph match: altitude
  Match method: high

Angle
  Source phrase found: True
  Graph match: angle
  Match method: high

Angle_bisector_theorem
  Source phrase found: True
  Graph match: angle bisector theorem
  Match method: high

Arc_(geometry)
  Source phrase found: True
  Graph match: arc
  Match method: high

Arc_length
  Source

In [ ]:

%cd /content/LearnKG

from pathlib import Path
import re

source_path = Path(
    "data_input/ck12_geometry/ck12_basic_geometry_concepts.txt"
)
text = source_path.read_text(encoding="utf-8", errors="replace")

terms_to_check = {
    "Collinearity": [
        "collinear", "collinearity", "betweenness"
    ],
    "Concentric": ["concentric", "same center", "common center"],
    "Convex_polygon": ["convex polygon", "convex", "polygon"],
    "Coordinate_system": [
        "coordinate system", "coordinate plane",
        "coordinates", "Cartesian"
    ],
    "Decagon": ["decagon", "10-sided", "ten-sided"],
    "Dodecagon": ["dodecagon", "12-sided", "twelve-sided"],
    "Edge_(geometry)": ["edge", "edges"],
    "Face_(geometry)": ["face", "faces"],
    "Geometry": ["geometry"],
    "Heptagon": ["heptagon", "7-sided", "seven-sided"],
    "Inverse_trigonometric_functions": [
        "inverse trigonometric", "inverse sine",
        "arcsin", "arccos", "arctan"
    ],
    "Nonagon": ["nonagon", "9-sided", "nine-sided"],
    "Octagon": ["octagon", "8-sided", "eight-sided"],
    "Parallel_(geometry)": ["parallel", "parallel lines"],
    "Perpendicular": ["perpendicular"],
    "Similarity_(geometry)": [
        "similarity", "similar polygons", "similar triangles"
    ],
    "Sine": ["sine", "sin("],
    "Skew_lines": ["skew lines", "skew"],
    "Tessellation": ["tessellation", "tessellate", "tiling"],
}

lines = text.splitlines()

for concept, terms in terms_to_check.items():
    hits = []

    for i, line in enumerate(lines):
        if any(
            re.search(r"(?<!\w)" + re.escape(term) + r"(?!\w)",
                      line, re.IGNORECASE)
            for term in terms
        ):
            hits.append((i + 1, line.strip()))

    print(f"\n{'=' * 70}\n{concept}: {len(hits)} matching lines")

    for line_number, line in hits[:8]:
        print(f"  L{line_number}: {line[:350]}")

    if len(hits) > 8:
        print(f"  ... and {len(hits) - 8} more lines")


/content/LearnKG

Collinearity: 2 matching lines
  L5: A point is a location with no dimension. It is represented by a dot. A line is a straight one-dimensional figure with no thickness that extends infinitely in both directions. A plane is a flat two-dimensional surface that extends infinitely in all directions. Points, lines, and planes are the three undefined terms of geometry. A line segment is a p
  L201: A glide reflection is the composition of a reflection and a translation parallel to the line of reflection. Isometries preserve angle measure, betweenness, collinearity, and distance. Any rigid transformation can be expressed as the composition of reflections.

Concentric: 0 matching lines

Convex_polygon: 4 matching lines
  L20: A polygon is a closed plane figure with three or more straight sides. Polygons are classified by the number of sides. A triangle has three sides. A quadrilateral has four sides. A pentagon has five sides. A hexagon has six sides. A regular polygon has al

In [ ]:
%cd /content/LearnKG

from pathlib import Path
import importlib
import pandas as pd
import evaluate_prerequisite_extraction as ev

# --------------------------------------------------
# A. Safely add mappings to the actual source file
# --------------------------------------------------
source_file = Path("evaluate_prerequisite_extraction.py")
source = source_file.read_text(encoding="utf-8")

anchor = '    "Two-dimensional space": ["Coordinate Plane", "Plane"],'

new_entries = {
    "Collinearity": '    "Collinearity": ["Collinear points"],',
    "Sine": '    "Sine": ["Sine Ratios"],',
    "Parallel": '    "Parallel": ["Parallel lines"],',
    "Perpendicular": '    "Perpendicular": ["Perpendicular lines"],',
}

missing_entries = [
    line for key, line in new_entries.items()
    if f'"{key}"' not in source
]

if missing_entries:
    if anchor not in source:
        raise RuntimeError(
            "Safety stop: expected dictionary anchor not found. "
            "No source changes were made."
        )

    insertion = "\n" + "\n".join(missing_entries)
    source = source.replace(anchor, anchor + insertion, 1)
    source_file.write_text(source, encoding="utf-8")
    print("Added mappings:", len(missing_entries))
else:
    print("All four mapping keys already exist; no source edit needed.")

# --------------------------------------------------
# B. Reload and test actual matching
# --------------------------------------------------
importlib.invalidate_caches()
ev = importlib.reload(ev)

concepts_path = Path(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv"
)
concepts_df = pd.read_csv(concepts_path).fillna("")

concepts_by_id, term_to_cid, normalizer = (
    ev.build_extracted_concept_index(concepts_df)
)

print("\n=== MATCHER VERIFICATION ===")

for gold in [
    "Collinearity",
    "Sine",
    "Parallel_(geometry)",
    "Perpendicular",
]:
    result = ev.match_gold_concept(
        gold, term_to_cid, concepts_by_id, normalizer
    )
    cid, method, score, notes = result
    name = concepts_by_id[cid]["concept_name"] if cid else None

    print({
        "gold": gold,
        "matched_id": cid,
        "matched_concept": name,
        "method": method,
        "score": score,
        "notes": notes,
    })

# --------------------------------------------------
# C. Inspect existing graph artifact schemas
# --------------------------------------------------
output_dir = Path(
    "data_output/ck12_geometry_kgx_v2_hybrid_k8"
)

print("\n=== CSV FILES AND SCHEMAS ===")

for path in sorted(output_dir.glob("*.csv")):
    try:
        df = pd.read_csv(path, nrows=5)
        print(f"\n{path.name}")
        print("Columns:", list(df.columns))
        print(df.head(2).to_string(index=False))
    except Exception as exc:
        print(f"\nCould not inspect {path.name}: {exc}")

# --------------------------------------------------
# D. Inspect the four important existing nodes
# --------------------------------------------------
print("\n=== EXISTING CONCEPT RECORDS ===")

focus_names = [
    "polygon",
    "Similar Polygons",
    "angle angle similarity postulate",
    "Sine Ratios",
    "Trigonometry",
    "CK-12 Basic Geometry Concepts",
]

for name in focus_names:
    rows = concepts_df[
        concepts_df["concept_name"].astype(str).str.casefold()
        == name.casefold()
    ]
    if not rows.empty:
        print(f"\n{name}")
        print(rows.to_string(index=False))

/content/LearnKG


RuntimeError: Safety stop: expected dictionary anchor not found. No source changes were made.

In [ ]:
from pathlib import Path

path = Path("/content/LearnKG/evaluate_prerequisite_extraction.py")

print("File exists:", path.exists())
print("Resolved path:", path.resolve() if path.exists() else "NOT FOUND")

if not path.exists():
    print("\nSearching for the actual file...")
    for p in Path("/content/LearnKG").rglob("evaluate_prerequisite_extraction.py"):
        print(p)
else:
    source = path.read_text(encoding="utf-8")
    lines = source.splitlines()

    # Show the synonym dictionary and its surrounding lines.
    hits = [
        i for i, line in enumerate(lines)
        if "GEOMETRY_SYNONYM_MAP" in line
        or "Two-dimensional space" in line
        or "Coordinate system" in line
        or "Skew lines" in line
    ]

    print("\nRelevant source lines:")
    for i in hits:
        start = max(0, i - 3)
        end = min(len(lines), i + 5)
        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]!r}")

File exists: True
Resolved path: /content/LearnKG/evaluate_prerequisite_extraction.py

Relevant source lines:

--- Lines 33-40 ---
  33: ''
  34: '# Mapping from AL-CPL / Wikipedia concept names to the terms used in CK-12 Basic Geometry.'
  35: '# These handle the 16 unmatched gold concepts and other known surface-form mismatches.'
  36: 'GEOMETRY_SYNONYM_MAP: Dict[str, List[str]] = {'
  37: '    # Direct AL-CPL → CK-12 name mappings'
  38: '    "Acute and obtuse triangles": ["Acute Triangle", "Obtuse Triangle"],'
  39: '    "Bisection":                  ["Perpendicular Bisector", "Angle Bisector", "Bisector"],'
  40: '    "Cartesian coordinate system":["Coordinate Plane", "Coordinate System", "Cartesian Coordinate"],'

--- Lines 38-45 ---
  38: '    "Acute and obtuse triangles": ["Acute Triangle", "Obtuse Triangle"],'
  39: '    "Bisection":                  ["Perpendicular Bisector", "Angle Bisector", "Bisector"],'
  40: '    "Cartesian coordinate system":["Coordinate Plane", "Coordi

In [ ]:
from pathlib import Path
import re
import shutil
import importlib
import sys

repo = Path("/content/LearnKG")
path = repo / "evaluate_prerequisite_extraction.py"

assert path.exists(), f"Source file not found: {path}"

source = path.read_text(encoding="utf-8")

# Locate the dictionary boundaries.
start_match = re.search(
    r"(?m)^GEOMETRY_SYNONYM_MAP\s*:\s*Dict\[.*?\]\s*=\s*\{",
    source
)
assert start_match, "Could not locate GEOMETRY_SYNONYM_MAP declaration."

dict_start = start_match.start()
open_brace = source.find("{", start_match.start())
close_brace = source.find("\n}", open_brace)

assert open_brace != -1 and close_brace != -1, (
    "Could not safely locate the dictionary's closing brace. No changes made."
)

dictionary = source[open_brace:close_brace + 2]

# Add only entries that do not already exist as dictionary keys.
entries = {
    "Collinearity": ["Collinear points"],
    "Sine": ["Sine Ratios"],
    "Parallel": ["Parallel lines"],
    "Perpendicular": ["Perpendicular lines"],
}

missing = {}
for key, values in entries.items():
    # Match a complete quoted key at the beginning of a dictionary entry.
    key_pattern = re.compile(
        r'(?m)^\s*["\']' + re.escape(key) + r'["\']\s*:'
    )
    if not key_pattern.search(dictionary):
        missing[key] = values

print("Mappings to add:", missing)

if missing:
    # Create a backup only when a change is needed.
    backup = path.with_suffix(".py.before_synonym_patch.bak")
    if not backup.exists():
        shutil.copy2(path, backup)
        print("Backup created:", backup)
    else:
        print("Existing backup preserved:", backup)

    new_lines = []
    for key, values in missing.items():
        new_lines.append(f"    {key!r}: {values!r},")

    # Insert before the closing brace.
    updated_dictionary = (
        dictionary[:-1]
        + "\n"
        + "\n".join(new_lines)
        + "\n}"
    )

    updated_source = (
        source[:open_brace]
        + updated_dictionary
        + source[close_brace + 2:]
    )

    # Syntax-check before writing anything.
    compile(updated_source, str(path), "exec")

    path.write_text(updated_source, encoding="utf-8")
    print("Source file updated.")
else:
    print("All four keys already exist; no source edit needed.")

# Verify Python syntax on disk.
compile(path.read_text(encoding="utf-8"), str(path), "exec")
print("Syntax check: PASS")

# Reload the module so tests use the updated source.
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import evaluate_prerequisite_extraction as ev
importlib.invalidate_caches()
ev = importlib.reload(ev)

print("\nLoaded synonym-map entries:")
for key in entries:
    print(f"{key!r} -> {ev.GEOMETRY_SYNONYM_MAP.get(key)}")

Mappings to add: {'Collinearity': ['Collinear points'], 'Sine': ['Sine Ratios'], 'Parallel': ['Parallel lines'], 'Perpendicular': ['Perpendicular lines']}
Backup created: /content/LearnKG/evaluate_prerequisite_extraction.py.before_synonym_patch.bak
Source file updated.
Syntax check: PASS

Loaded synonym-map entries:
'Collinearity' -> ['Collinear points']
'Sine' -> ['Sine Ratios']
'Parallel' -> ['Parallel lines']
'Perpendicular' -> ['Perpendicular lines']


In [ ]:
from pathlib import Path
import pandas as pd
import evaluate_prerequisite_extraction as ev

kg_dir = Path(
    "/content/LearnKG/data_output/ck12_geometry_kgx_v2_hybrid_k8"
)
concepts_path = kg_dir / "concepts.csv"

assert concepts_path.exists(), f"Missing concepts file: {concepts_path}"

concepts = pd.read_csv(concepts_path)

concepts_by_id, term_to_cid, normalizer = (
    ev.build_extracted_concept_index(concepts)
)

tests = [
    "Collinearity",
    "Sine",
    "Parallel_(geometry)",
    "Perpendicular",
]

print("=== MATCHING RESULTS ===")
for gold in tests:
    result = ev.match_gold_concept(
        gold,
        term_to_cid,
        concepts_by_id,
        normalizer,
        fuzzy_threshold=0.82,
    )
    cid, confidence, score, notes = result

    node_name = (
        concepts_by_id[cid].get("concept_name")
        if cid is not None
        else None
    )

    print(f"\nGold concept : {gold}")
    print(f"Matched ID   : {cid}")
    print(f"Matched name : {node_name}")
    print(f"Confidence   : {confidence}")
    print(f"Score        : {score}")
    print(f"Candidates   : {notes}")

=== MATCHING RESULTS ===

Gold concept : Collinearity
Matched ID   : C0036
Matched name : Collinear points
Confidence   : high
Score        : 0.97
Candidates   : ["Synonym map: 'Collinear points' matches 'Collinear points'"]

Gold concept : Sine
Matched ID   : C0171
Matched name : Sine Ratios
Confidence   : high
Score        : 0.97
Candidates   : ["Synonym map: 'Sine Ratios' matches 'Sine Ratios'"]

Gold concept : Parallel_(geometry)
Matched ID   : C0122
Matched name : Parallel lines
Confidence   : high
Score        : 0.97
Candidates   : ["Synonym map: 'Parallel lines' matches 'Parallel lines'"]

Gold concept : Perpendicular
Matched ID   : C0129
Matched name : Perpendicular lines
Confidence   : high
Score        : 0.97
Candidates   : ["Synonym map: 'Perpendicular lines' matches 'Perpendicular lines'"]


In [ ]:
from pathlib import Path
import pandas as pd
import re

repo = Path("/content/LearnKG")
kg_dir = repo / "data_output/ck12_geometry_kgx_v2_hybrid_k8"
text_path = repo / "data_input/ck12_geometry/ck12_basic_geometry_concepts.txt"

# --------------------------------------------------
# 1. Inspect graph file schemas
# --------------------------------------------------
print("=" * 70)
print("1. GRAPH FILES AND SCHEMAS")
print("=" * 70)

for path in sorted(kg_dir.glob("*.csv")):
    try:
        df = pd.read_csv(path, nrows=3)
        print(f"\n{path.name}")
        print("Columns:", df.columns.tolist())
        print("Sample rows:")
        print(df.to_string(index=False, max_colwidth=70))
    except Exception as e:
        print(f"\nCould not inspect {path.name}: {e}")

# --------------------------------------------------
# 2. Inspect existing concept nodes
# --------------------------------------------------
print("\n" + "=" * 70)
print("2. EXISTING RELATED CONCEPT NODES")
print("=" * 70)

concepts_path = kg_dir / "concepts.csv"
concepts = pd.read_csv(concepts_path).fillna("")

keywords = [
    "convex", "polygon", "inverse trigonometric",
    "similarity", "similar polygons",
    "sine", "trigonometry", "collinear",
    "parallel", "perpendicular", "geometry"
]

for _, row in concepts.iterrows():
    name = str(row.get("concept_name", ""))
    aliases = str(row.get("aliases", ""))
    definition = str(row.get("definition", ""))

    searchable = f"{name} {aliases} {definition}".lower()

    if any(keyword in searchable for keyword in keywords):
        print(
            f"\nID: {row.get('concept_id', '')}"
            f"\nName: {name}"
            f"\nAliases: {aliases}"
            f"\nDefinition: {definition}"
        )

# --------------------------------------------------
# 3. Search the original textbook for supporting evidence
# --------------------------------------------------
print("\n" + "=" * 70)
print("3. TEXTBOOK EVIDENCE")
print("=" * 70)

assert text_path.exists(), f"Textbook file not found: {text_path}"
lines = text_path.read_text(encoding="utf-8").splitlines()

patterns = {
    "Convex polygon": r"\bconvex polygon\b|\bconvex polygons\b",
    "Inverse trigonometric functions":
        r"\binverse trigonometric functions?\b",
    "Similarity": r"\bsimilarity\b|\bsimilar polygons?\b",
    "Geometry": r"\bgeometry\b",
    "Sine": r"\bsine\b",
    "Collinearity": r"\bcollinearity\b|\bcollinear\b",
    "Parallel": r"\bparallel lines?\b",
    "Perpendicular": r"\bperpendicular\b",
}

for label, pattern in patterns.items():
    print(f"\n--- {label} ---")
    hits = [
        (i, line.strip())
        for i, line in enumerate(lines)
        if re.search(pattern, line, flags=re.IGNORECASE)
    ]

    if not hits:
        print("No literal match found.")
        continue

    for line_number, line in hits[:8]:
        print(f"Line {line_number + 1}: {line[:500]}")

    if len(hits) > 8:
        print(f"... {len(hits) - 8} additional matching lines omitted.")

print("\nInspection complete. No files were modified.")

1. GRAPH FILES AND SCHEMAS

concepts.csv
Columns: ['concept_id', 'concept_name', 'aliases', 'concept_type', 'definition', 'chapter', 'section', 'page', 'chunk_id', 'evidence']
Sample rows:
concept_id    concept_name  aliases concept_type                                                             definition                       chapter                   section  page                         chunk_id                                                               evidence
     C0001     acute angle      NaN      Concept                             Acute angles measure less than 90 degrees. Chapter 1: Basics of Geometry       1.3 Rays and Angles     1 a6963ad59e6b4ecdb28d588d3d4ed83f                             Acute angles measure less than 90 degrees.
     C0002  acute triangle      NaN      Concept                 An acute triangle has all angles less than 90 degrees. Chapter 1: Basics of Geometry 1.5 Classifying Triangles     2 359afd06a29d48a2920c5a35ac7b1780                 An acute

In [ ]:
from pathlib import Path
import pandas as pd
import re

kg_dir = Path(
    "/content/LearnKG/data_output/ck12_geometry_kgx_v2_hybrid_k8"
)

concepts = pd.read_csv(kg_dir / "concepts.csv").fillna("")
relations = pd.read_csv(kg_dir / "relations.csv").fillna("")

print("=" * 65)
print("A. POSSIBLE DUPLICATES")
print("=" * 65)

queries = {
    "Convex polygon": ["convex polygon", "convex polygons"],
    "Inverse trigonometric functions": [
        "inverse trigonometric functions",
        "inverse trigonometric function",
        "arcsine",
        "arccosine",
        "arctangent",
    ],
    "General similarity": [
        "similarity",
        "similar polygons",
        "similarity theorem",
        "similarity postulate",
    ],
}

for label, terms in queries.items():
    print(f"\n--- {label} ---")
    found = False

    for _, row in concepts.iterrows():
        searchable = " ".join([
            str(row.get("concept_name", "")),
            str(row.get("aliases", "")),
            str(row.get("definition", "")),
            str(row.get("evidence", "")),
        ]).lower()

        matched_terms = [t for t in terms if t in searchable]
        if matched_terms:
            found = True
            print({
                "concept_id": row["concept_id"],
                "concept_name": row["concept_name"],
                "aliases": row["aliases"],
                "matched_terms": matched_terms,
                "chapter": row["chapter"],
                "section": row["section"],
                "page": row["page"],
                "chunk_id": row["chunk_id"],
                "evidence": str(row["evidence"])[:250],
            })

    if not found:
        print("No matching existing node.")

print("\n" + "=" * 65)
print("B. SOURCE-NODE PROVENANCE")
print("=" * 65)

for cid in ["C0075", "C0170", "C0010", "C0171", "C0185", "C0197"]:
    rows = concepts[concepts["concept_id"].astype(str) == cid]
    if not rows.empty:
        print("\n", rows.to_dict(orient="records")[0])

print("\n" + "=" * 65)
print("C. CURRENT ID RANGES")
print("=" * 65)

def numeric_suffix(value):
    match = re.search(r"(\d+)$", str(value))
    return int(match.group(1)) if match else -1

concept_ids = concepts["concept_id"].astype(str)
relation_ids = relations["relation_id"].astype(str)

print("Concept count:", len(concepts))
print("Duplicate concept IDs:", concepts["concept_id"].duplicated().sum())
print("Highest concept ID:", max(concept_ids, key=numeric_suffix))
print("Relation count:", len(relations))
print("Duplicate relation IDs:", relations["relation_id"].duplicated().sum())
print("Highest relation ID:", max(relation_ids, key=numeric_suffix))

print("\nNo files were modified.")

A. POSSIBLE DUPLICATES

--- Convex polygon ---
{'concept_id': 'C0075', 'concept_name': 'exterior angle sum', 'aliases': '', 'matched_terms': ['convex polygon'], 'chapter': 'Chapter 6: Polygons and Quadrilaterals', 'section': '6.1 Angles in Polygons', 'page': 5, 'chunk_id': 'ab0ae1e2b0fb4342a2255056a72784f9', 'evidence': 'The exterior angle sum of any convex polygon is always 360 degrees.'}

--- Inverse trigonometric functions ---
No matching existing node.

--- General similarity ---
{'concept_id': 'C0010', 'concept_name': 'angle angle similarity postulate', 'aliases': '', 'matched_terms': ['similarity', 'similarity postulate'], 'chapter': 'Chapter 7: Similarity', 'section': '7.3 Similarity by AA, SSS, SAS', 'page': 6, 'chunk_id': '5a5863f3d3f2426499dbe24c7aa4aa50', 'evidence': 'The angle angle similarity postulate states that if two angles of one triangle are congruent to two angles of another, then the triangles are similar.'}
{'concept_id': 'C0125', 'concept_name': 'perimeter', 'ali

In [ ]:
%cd /content/LearnKG

!python -m kg_eval.prereq_gold --kg-dir data_output/ck12_geometry_kgx_v2_hybrid_k8

/content/LearnKG
Gold concepts matched to KG: 69/84;  pairs matched: 1113/1595;  located in text: 1431

MAIN (matched & located pairs)  (n=1113, positives=316, prevalence=0.284)
  predictor             P      R     F1  F1 95% CI          MCC  ROC-AUC  AUC 95% CI       PR-AUC
  kg_prereq         0.770  0.370  0.500  [0.44, 0.55]     0.429    0.666  [0.64, 0.69]      0.502
  kg_closure        0.676  0.873  0.762  [0.73, 0.80]     0.662    0.875  [0.85, 0.90]      0.690
  kg_any_relation   0.756  0.373  0.500  [0.44, 0.55]     0.423    0.666  [0.64, 0.69]      0.499
  kg_undirected     0.373  0.380  0.376  [0.33, 0.42]     0.126    0.563  [0.53, 0.59]      0.318
  kg_pairscore      0.914  0.168  0.283  [0.22, 0.34]     0.328    0.863  [0.84, 0.89]      0.702
  book_order        0.546  0.949  0.694  [0.66, 0.73]     0.574    0.852  [0.83, 0.87]      0.597
  cooccurrence      0.356  0.329  0.342  [0.29, 0.39]     0.096    0.549  [0.52, 0.58]      0.318
  cooc_x_order      0.698  0.307  0.42

In [ ]:
import json
from pathlib import Path

report_path = Path(
    "/content/LearnKG/data_output/"
    "ck12_geometry_kgx_v2_hybrid_k8/kg_eval/"
    "prereq_gold_report.json"
)

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

print("Report keys:", list(report.keys()))

matches = report.get("concept_matches", {})

priority_concepts = [
    "Collinearity",
    "Convex_polygon",
    "Geometry",
    "Inverse_trigonometric_functions",
    "Parallel_(geometry)",
    "Perpendicular",
    "Similarity_(geometry)",
    "Sine",
]

print("\nPRIORITY CONCEPT MATCHING RESULTS")
print("=" * 70)

for gold_name in priority_concepts:
    result = matches.get(gold_name)

    # Try case-insensitive matching if the exact key is absent.
    if result is None and isinstance(matches, dict):
        result = next(
            (
                value
                for key, value in matches.items()
                if key.lower() == gold_name.lower()
            ),
            None,
        )

    print(f"\nGold concept: {gold_name}")
    print("Match:", result if result is not None else "No entry found")

Report keys: ['kg_dir', 'gold', 'gold_pairs', 'gold_positive', 'gold_concepts', 'gold_concepts_matched_to_kg', 'gold_concepts_low_confidence_match', 'gold_concepts_unmatched', 'pairs_matched', 'pairs_located_in_text', 'main', 'end_to_end', 'kg_prereq_weight_only', 'kg_prereq_edges_reversed_vs_gold', 'concept_matches', 'notes']

PRIORITY CONCEPT MATCHING RESULTS

Gold concept: Collinearity
Match: {'cid': 'C0036', 'kg_name': 'Collinear points', 'confidence': 'high'}

Gold concept: Convex_polygon
Match: {'cid': None, 'kg_name': None, 'confidence': 'unmatched'}

Gold concept: Geometry
Match: {'cid': None, 'kg_name': None, 'confidence': 'unmatched'}

Gold concept: Inverse_trigonometric_functions
Match: {'cid': None, 'kg_name': None, 'confidence': 'unmatched'}

Gold concept: Parallel_(geometry)
Match: {'cid': 'C0122', 'kg_name': 'Parallel lines', 'confidence': 'high'}

Gold concept: Perpendicular
Match: {'cid': 'C0129', 'kg_name': 'Perpendicular lines', 'confidence': 'high'}

Gold concept: S

In [ ]:
from pathlib import Path
import pandas as pd
import re

kg_dir = Path(
    "/content/LearnKG/data_output/"
    "ck12_geometry_kgx_v2_hybrid_k8"
)

concepts = pd.read_csv(kg_dir / "concepts.csv").fillna("")

search_patterns = {
    "Convex polygon": r"\bconvex\b|\bpolygon\b",
    "Geometry": r"\bgeometry\b|\bgeometric\b",
    "Inverse trigonometric functions": (
        r"inverse trig|arcsin|arccos|arctan|"
        r"arc sine|arc cosine|arc tangent"
    ),
    "Similarity": r"\bsimilar\b|\bsimilarity\b",
}

for label, pattern in search_patterns.items():
    print(f"\n{'=' * 80}")
    print(f"SEARCH: {label}")
    print("=" * 80)

    mask = concepts.apply(
        lambda row: bool(
            re.search(
                pattern,
                " ".join(str(value) for value in row.values),
                flags=re.IGNORECASE,
            )
        ),
        axis=1,
    )

    columns = [
        "concept_id",
        "concept_name",
        "aliases",
        "definition",
        "chapter",
        "section",
        "evidence",
    ]

    results = concepts.loc[mask, columns]

    if results.empty:
        print("No matching nodes found.")
    else:
        print(results.to_string(index=False))


SEARCH: Convex polygon
concept_id           concept_name     aliases                                                                                                         definition                                chapter                           section                                                                                                           evidence
     C0075     exterior angle sum                                                            The exterior angle sum of any convex polygon is always 360 degrees. Chapter 6: Polygons and Quadrilaterals            6.1 Angles in Polygons                                                The exterior angle sum of any convex polygon is always 360 degrees.
     C0089     interior angle sum                                               The interior angle sum of a polygon with n sides is n minus 2 times 180 degrees. Chapter 6: Polygons and Quadrilaterals            6.1 Angles in Polygons                                   The interi

In [ ]:
import json
from pathlib import Path

report_path = Path(
    "/content/LearnKG/data_output/"
    "ck12_geometry_kgx_v2_hybrid_k8/kg_eval/"
    "prereq_gold_report.json"
)

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

unmatched = report.get("gold_concepts_unmatched", [])

print("UNMATCHED GOLD CONCEPTS")
print("=" * 60)

if isinstance(unmatched, dict):
    for name, details in unmatched.items():
        print(f"{name}: {details}")
elif isinstance(unmatched, list):
    for item in unmatched:
        print(item)
else:
    print("Unexpected format:", type(unmatched).__name__)

UNMATCHED GOLD CONCEPTS
Concentric
Convex_polygon
Coordinate_system
Decagon
Dodecagon
Edge_(geometry)
Face_(geometry)
Geometry
Heptagon
Inverse_trigonometric_functions
Nonagon
Octagon
Similarity_(geometry)
Skew_lines
Tessellation


In [ ]:
from pathlib import Path
import pandas as pd
import re

kg_dir = Path(
    "/content/LearnKG/data_output/"
    "ck12_geometry_kgx_v2_hybrid_k8"
)

concepts = pd.read_csv(kg_dir / "concepts.csv").fillna("")

targets = {
    "Concentric": r"\bconcentric\b",
    "Coordinate_system": r"coordinate system|coordinate plane|cartesian",
    "Decagon": r"\bdecagon\b|\b10[- ]?gon\b",
    "Dodecagon": r"\bdodecagon\b|\b12[- ]?gon\b",
    "Edge_(geometry)": r"\bedges?\b",
    "Face_(geometry)": r"\bfaces?\b",
    "Heptagon": r"\bheptagon\b|\b7[- ]?gon\b",
    "Nonagon": r"\bnonagon\b|\b9[- ]?gon\b",
    "Octagon": r"\boctagon\b|\b8[- ]?gon\b",
    "Skew_lines": r"\bskew lines?\b",
    "Tessellation": r"\btessellations?\b",
}

search_columns = [
    "concept_name",
    "aliases",
    "definition",
    "evidence",
]

output_columns = [
    "concept_id",
    "concept_name",
    "aliases",
    "definition",
    "chapter",
    "section",
    "evidence",
]

for target, pattern in targets.items():
    print("\n" + "=" * 85)
    print(f"TARGET: {target}")
    print("=" * 85)

    # Search each field independently so the results are easy to audit.
    mask = concepts[search_columns].apply(
        lambda col: col.astype(str).str.contains(
            pattern,
            case=False,
            regex=True,
            na=False,
        )
    ).any(axis=1)

    results = concepts.loc[mask, output_columns]

    if results.empty:
        print("No matching terms found in existing graph nodes.")
    else:
        print(results.to_string(index=False))


TARGET: Concentric
No matching terms found in existing graph nodes.

TARGET: Coordinate_system
No matching terms found in existing graph nodes.

TARGET: Decagon
No matching terms found in existing graph nodes.

TARGET: Dodecagon
No matching terms found in existing graph nodes.

TARGET: Edge_(geometry)
concept_id concept_name     aliases                                                                                                         definition                             chapter          section                                                                                                           evidence
     C0136   polyhedron Polyhedrons 11.1 Polyhedrons A polyhedron is a three-dimensional figure with flat polygon faces, straight edges, and vertices. Chapter 11: Surface Area and Volume 11.1 Polyhedrons 11.1 Polyhedrons A polyhedron is a three-dimensional figure with flat polygon faces, straight edges, and vertices.
     C0202     vertices             11.1 Polyhedrons A pol

In [ ]:
from pathlib import Path
import hashlib

root = Path("/content/LearnKG")

files = [
    root / "evaluate_prerequisite_extraction.py",
    root / "data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv",
    root / "data_output/ck12_geometry_kgx_v2_hybrid_k8/relations.csv",
    root / "data_output/ck12_geometry_kgx_v2_hybrid_k8/pair_scores.csv",
    root / "data_output/ck12_geometry_kgx_v2_hybrid_k8/kg_eval/prereq_gold_report.json",
]

print("CURRENT CHECKPOINT")
print("=" * 75)

for path in files:
    if path.exists():
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        print(f"\nFile: {path.relative_to(root)}")
        print(f"Size: {path.stat().st_size:,} bytes")
        print(f"SHA-256: {digest}")
    else:
        print(f"\nMISSING: {path.relative_to(root)}")

CURRENT CHECKPOINT

File: evaluate_prerequisite_extraction.py
Size: 21,717 bytes
SHA-256: d219a83853ed3b9a6775ca5c1f3f6d79878ceebecac2903da1f7d6db119e6168

File: data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv
Size: 61,846 bytes
SHA-256: 293c23af113f855a809ac659b0525f572e63ed4ac535b4900b377c372d97abef

File: data_output/ck12_geometry_kgx_v2_hybrid_k8/relations.csv
Size: 170,746 bytes
SHA-256: bacb1313985cdeaf3ba56d4eff2becef6c6ceb5dc69d0252f8643ce20740c4cd

File: data_output/ck12_geometry_kgx_v2_hybrid_k8/pair_scores.csv
Size: 2,898,869 bytes
SHA-256: b413b2172bab70a13e809d4a2f8df3f1af43f99b4407f74fd2208068db2b78f9

File: data_output/ck12_geometry_kgx_v2_hybrid_k8/kg_eval/prereq_gold_report.json
Size: 17,340 bytes
SHA-256: af18adcefaecba6e5b733f8201c15d20d10994e8cc5b2e9b641c16e90fd4ecb2


In [ ]:
from pathlib import Path
import zipfile

root = Path("/content/LearnKG")
kg_dir = root / "data_output/ck12_geometry_kgx_v2_hybrid_k8"

files_to_package = [
    kg_dir / "concepts.csv",
    kg_dir / "relations.csv",
    kg_dir / "pair_scores.csv",
    kg_dir / "kg_eval/prereq_gold_report.json",
]

# Include the updated evaluator so the local evaluation can use
# the same synonym mappings as the latest Colab evaluation.
files_to_package.append(root / "evaluate_prerequisite_extraction.py")

missing = [str(p) for p in files_to_package if not p.is_file()]

if missing:
    print("STOP — these expected files are missing:")
    for p in missing:
        print("  ", p)
else:
    zip_path = root / "LearnKG_latest_evaluation_artifacts.zip"

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in files_to_package:
            zf.write(path, path.relative_to(root))

    print("Created:", zip_path)
    print("Files included:")
    for path in files_to_package:
        print(" -", path.relative_to(root))

Created: /content/LearnKG/LearnKG_latest_evaluation_artifacts.zip
Files included:
 - data_output/ck12_geometry_kgx_v2_hybrid_k8/concepts.csv
 - data_output/ck12_geometry_kgx_v2_hybrid_k8/relations.csv
 - data_output/ck12_geometry_kgx_v2_hybrid_k8/pair_scores.csv
 - data_output/ck12_geometry_kgx_v2_hybrid_k8/kg_eval/prereq_gold_report.json
 - evaluate_prerequisite_extraction.py


In [ ]:
from google.colab import files
files.download("/content/LearnKG/LearnKG_latest_evaluation_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>